In [ ]:
# --- Importing Libraries
import sys
print("Python Executable:", sys.executable)

import pandas as pd # type: ignore
import numpy as np

np.int = int # 😠 come on pygam
import matplotlib.pyplot as plt # type: ignore
import seaborn as sns # type: ignore
import statsmodels.api as sm # type: ignore
import statsmodels.formula.api as smf # type: ignore
import matplotlib.pyplot as plt # type: ignore
from stargazer.stargazer import Stargazer # type: ignore
from IPython.display import HTML, display # type: ignore
from sklearn.preprocessing import MinMaxScaler # type: ignore
import re
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.utils import resample
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.inspection import PartialDependenceDisplay
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.utils.validation import check_is_fitted
from sklearn.gaussian_process.kernels import Matern
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from pygam import LinearGAM, s
import pymc as pm
#Hacking together version deprecationsin pygam
import scipy.sparse
scipy.sparse.csr.csr_matrix.A = property(lambda self: self.toarray())


plt.style.use('dark_background')


# Create a font object
from matplotlib import font_manager as fm
poiret = fm.FontProperties(fname="C:/Users/ben.pharris/Downloads/Poiret_One/Montserrat-SemiBold.ttf",size=None)
from matplotlib import font_manager
font_manager.fontManager.addfont("C:/Users/ben.pharris/Downloads/Poiret_One/Montserrat-SemiBold.ttf")

plt.rcParams['font.family'] = poiret.get_name()

del(poiret)

In [ ]:
# --- Read in Data, clean columns, convert date to datetime
file_path = "../data/processed/mkt_2h_2024.csv"
df = pd.read_csv(file_path)
del file_path

df['Date'] = pd.to_datetime(df['Date'])

df = df.drop(columns=['Month', 'Indirect Traffic', 'Digital Traffic', 'DS', 'Door Count', 'CPIAUCSL'])

df.info()


In [ ]:
# --- Renaming variables using a function + regex

def to_snake_case(s: str) -> str:
    # 1. Insert underscore before an uppercase letter that follows a lowercase letter (handle camelCase)
    s = re.sub(r'(?<=[a-z])([A-Z])', r'_\1', s)
    
    # 2. Remove any non-alphanumeric (or underscore) characters that are sandwiched between allowed characters,
    #    but only if that group does not include a space.
    #    Allowed characters are letters, digits, or underscore (\w).
    s = re.sub(r'(?<=[\w])((?!\s)[^\w])+(?=[\w])', '', s)
    
    # 3. Replace any remaining sequence of characters that is not a letter, digit, or underscore with an underscore.
    s = re.sub(r'[^\w]+', '_', s)
    
    # 4. Lowercase the string and trim leading/trailing underscores.
    return s.lower().strip('_')

df.columns = [to_snake_case(col) for col in df.columns]


In [ ]:
# --- Define 2024 Sub Channels
df['paid_social'] = df[['tik_tok', 'snap', 'meta']].sum(axis=1)
df['paid_search'] = df[['google_search', 'microsoft_search']].sum(axis=1)
df['paid_shopping'] = df[['google_shopping', 'microsoft_shopping', 'amazon_display_dsp', 'amazon_search']].sum(axis=1)
df['email'] = df[['zeta_prepaid_indirect', 'zeta_digital']].sum(axis=1)
df['local'] = df[['i_heart', 'local_google_prepaid_indirect', 'local_marketing_dtrcommunity_events', 'local_meta_prepaid_indirect', 'national_retail' ]].sum(axis=1)
df['program'] = df[['affiliate', 'googleyou_tube', 'ovative_passthrough', 'univision', 'yahoo']].sum(axis=1)

df['indirect'] = df['ga_indirect'] + df['ga_retail']
df['ecom'] = df['ga_ecom'] + df['ga_retail']

In [ ]:
# --- Read in 2025 Data
test_data = pd.read_csv('C:/Users/ben.pharris/Downloads/Q12025.csv')
test_data['date'] = pd.to_datetime(test_data['date'])
test_data['total_spend'] = test_data[['paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']].sum(axis=1)


test_data['indirect'] = test_data['ga_indirect'] + test_data['ga_retail']
test_data['ecom'] = test_data['ga_ecom'] + test_data['ga_apple']
test_data.head()

In [ ]:
# --- Combined Analysis DF
df_2024 = df.copy()

subset_24 = df[['date', 'ga', 'paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program', 'ecom', 'indirect']]
subset_25Q1 = test_data[['date', 'ga', 'paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program', 'ecom', 'indirect']]

df = pd.concat([subset_24, test_data], ignore_index=True)

df_2025 = test_data.copy()
del(test_data)
del(subset_24)
del(subset_25Q1)

In [ ]:
# --- Date Effects

df['days_since'] = (df['date'] - df['date'].min()).dt.days  # numeric trend
df['day_of_week'] = df['date'].dt.weekday                  # categorical / cyclical
df['month'] = df['date'].dt.month
df['day_of_year'] = df['date'].dt.dayofyear

# Cyclical encoding for weekly or annual patterns
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

In [ ]:
# --- Drop NA
# 1) List the five columns you know you don’t need:
cols_to_drop = ['ga_indirect', 'ga_ecom', 'ga_retail', 'ga_apple', 'ga_amazon','ga_cross','ga-telesale','ga_other', 'total_spend']  # ← replace with your actual names

# 2) Drop them…
df = df.drop(columns=cols_to_drop)

# 3) …then drop any row that still has at least one NA
df = df.dropna(axis=0, how='any').reset_index(drop=True)

# 4) (Optional) Quick sanity check
print("Shape after cleaning:", df.shape)
print(df.isna().sum())   # should all be zero


In [ ]:
# --- integrity checks
print("Index unique? ", df.index.is_unique)

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

# 1) Define features & targets
feature_names = [
    'days_since',
    'paid_social',
    'paid_search',
    'paid_shopping',
    'email',
    'local',
    'program'
]
X = df[feature_names].values

targets = {
    'Indirect': 'indirect',
    'E-commerce': 'ecom'
}

# 2) Parameters for the GBM
gbm_params = dict(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42
)

# 3) Run through each target and collect metrics + importances
rows = []
for label, col in targets.items():
    y = df[col].values

    # 3a) Cross-validated R²
    model = GradientBoostingRegressor(**gbm_params)
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)

    # 3b) Fit on full data
    model.fit(X, y)
    train_r2 = model.score(X, y)
    mean_cv = np.mean(cv_scores)

    # 3c) Feature importances
    imp = pd.Series(model.feature_importances_, index=feature_names)

    # 3d) Build a single row of results
    row = {
        'Output':       label,
        'Train R²':     train_r2,
        'Mean CV R²':   mean_cv
    }
    # add each feature’s importance as its own column
    row.update({f'Imp: {f}': round(imp[f], 3) for f in feature_names})
    rows.append(row)

# 4) Assemble into a DataFrame
importance_df = pd.DataFrame(rows).set_index('Output')

# 5) (Optional) Re-order columns so R² metrics come first
cols = ['Train R²', 'Mean CV R²'] + [c for c in importance_df.columns if c.startswith('Imp: ')]
importance_df = importance_df[cols]

print(importance_df)

In [ ]:
import matplotlib.pyplot as plt

# 1) Pick out just the importance columns
imp_cols = [c for c in importance_df.columns if c.startswith('Imp: ')]
imp_df   = importance_df[imp_cols].copy()

# 2) Tidy up the names (strip off “Imp: ”)
imp_df.columns = [c.replace('Imp: ', '') for c in imp_df.columns]

# 3) Transpose so features→rows, outputs→columns
plot_df = imp_df.T

# 4) Plot
fig, ax = plt.subplots(figsize=(12, 6))
plot_df.plot(kind='bar', ax=ax)

ax.set_title('Feature Importances by Output', pad=16)
ax.set_xlabel('Feature')
ax.set_ylabel('Importance')
ax.legend(title='Output', loc='upper right')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()


In [ ]:
# --- Sales over Predictor Plots
predictors = ['paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']
outcome = 'ga'

# ChatGPT making the size of the grid dynamic
n_cols = 3
n_rows = int(np.ceil(len(predictors) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 10), constrained_layout=True)
axes = axes.flatten()

for i, predictor in enumerate(predictors):
    ax = axes[i]
    sns.scatterplot(data=df, x=predictor, y=outcome, ax=ax, s=20, alpha=0.7)
    #ax.set_title(f'{predictor} vs {outcome}', fontsize=12)
    ax.set_xlabel(predictor)
    ax.set_ylabel(outcome)

plt.suptitle('Scatterplots of Predictors vs Outcome', fontsize=16)
plt.show()

In [ ]:
# --- OLS Regressions
n = smf.ols("ga ~ days_since + paid_social + paid_search + paid_shopping + email + local + program", data = df).fit(cov_type='HC1')
stargazer = Stargazer([n])
html_output = stargazer.render_html()
display(HTML(html_output))

In [ ]:
# --- OLS visualization
'''
This is predictions of each B*x against x. Partial derivative plot of each term.
'''

predictor_cols = ['days_since', 'paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']

fig, axs = plt.subplots(2, 4, figsize=(15, 8))
axes = axs.flatten()

for ax, col in zip(axes, predictor_cols):
    effect_col = f'{col}_effect'
    
    # Calculate the effect
    df[effect_col] = n.params[col] * df[col]

    # Sort for smooth line
    sorted_df = df.sort_values(by=col)

    # Plot points
    ax.scatter(df[col], df[effect_col], alpha=0.4, label='Data Points')

    # Plot line
    ax.plot(sorted_df[col], sorted_df[effect_col], color='orange', label='Fitted Line', linewidth=1.5)

    # Optional formatting
    ax.axhline(0, linestyle='--', color='gray')
    #ax.set_title(f"Effect of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("βx Contribution")

for ax in axes[len(predictor_cols):]:
    ax.axis("off")

fig.tight_layout()
plt.show()

In [ ]:
# --- LASSO (Enet)

#Predictors and Outcome
X = df[['days_since','paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']].values
y = df['ga'].values

#Separate names because np array has no colnames
feature_names = df[['paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']].columns.tolist()

#Pipeline / Model
elasticnet = make_pipeline(
    StandardScaler(),
    ElasticNetCV(cv=5, l1_ratio=[.1, .5, .9, 1], random_state=42)
)

#Fit Model
elasticnet.fit(X, y)
predicted = elasticnet.predict(X)

# --- Extract scaler and model
scaler   = elasticnet.named_steps['standardscaler']
enet_cv  = elasticnet.named_steps['elasticnetcv']
β_scaled = enet_cv.coef_               
σ        = scaler.scale_              
μ = scaler.mean_
r2 = r2_score(y, predicted)
rmse = np.sqrt(root_mean_squared_error(y, predicted))

# --- Convert back to raw-scale coefficients
β_raw = β_scaled / σ  
intercept_raw = enet_cv.intercept_ - (β_scaled * μ / σ).sum()


# --- Compute per-variable effects in DataFrame
for coef, col in zip(β_raw, feature_names):
    df[f'{col}_effect'] = coef * df[col]


n_boot = 100
boot_coefs = []

# Coefficient table
for _ in range(n_boot):
    Xb, yb = resample(X, y, replace=True)
    elasticnet.fit(Xb, yb)
    en = elasticnet.named_steps['elasticnetcv']
    sc = elasticnet.named_steps['standardscaler']
    boot_coefs.append(en.coef_ / sc.scale_)

se_raw = np.std(boot_coefs, axis=0, ddof=1)

# Hacky Table
print("ElasticNet Results")
print("===================")
print(f"{'Variable':15s} {'Coef':>8s} {'Std.Err':>8s}")
for name, coef, se in zip(feature_names, β_raw, se_raw):
    print(f"{name:15s} {coef:8.4f} {se:8.4f}")

print(f"{'Intercept':15s} {intercept_raw:8.4f}")
print()
print(f"R²:   {r2:.4f}")
print(f"RMSE: {rmse:.2f}")


In [ ]:
# --- LASSO  Plots

def format_x(value, tick_number):
    if value >= 1_000_000:
        return f'${value/1_000_000:.1f}M'
    elif value >= 1_000:
        return f'${value/1_000:.1f}K'
    else:
        return f'${value:.0f}'

def format_y(value, tick_number):
    if abs(value) >= 1_000:
        return f'{value/1_000:.1f}K'
    else:
        return f'{value:.0f}'

formatter_x = FuncFormatter(format_x)
formatter_y = FuncFormatter(format_y)

# Create the grid
fig, axes = plt.subplots(2, 3, figsize=(20, 13), sharey=True)

# Loop over each subplot
for ax, col, coef in zip(axes.flatten(), feature_names, β_raw):
    # sort for smooth line
    sorted_idx = df[col].argsort()
    x_sorted   = df[col].iloc[sorted_idx]
    y_effect   = df[f'{col}_effect'].iloc[sorted_idx]

    ax.scatter(df[col], df[f'{col}_effect'], alpha=0.3)
    ax.plot(x_sorted, y_effect, linewidth=1.5,color='orange')

    ax.axhline(0, linestyle='--', color='gray')
    ax.set_title(f"Effect of {col}")

    # 4) Force an x‑ and y‑label on every subplot
    ax.set_xlabel(col)
    ax.set_ylabel("GAs")

    # 5) Apply our custom tick formatters
    ax.xaxis.set_major_formatter(formatter_x)
    ax.yaxis.set_major_formatter(formatter_y)

    # 6) Ensure y‑ticks & labels show up on all subplots
    ax.tick_params(labelleft=True)

# 7) Tidy up layout
fig.tight_layout()
plt.show()


In [ ]:
# --- GAM Regressions

X = df[['days_since', 'paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']].values
y = df['ga'].values


gam = LinearGAM(s(0) + s(1) + s(2) + s(3) + s(4) + s(5) + s(6)).fit(X, y)
gam.summary()

df['sales_gam'] = gam.predict(X)

In [ ]:
# --- Partial Dependence GAM plot

'''
Per chatGPT:
PDP's show the shape of the effect of each variable on the outcome, in the units of that variable,
which is one of the coolest things about GAMs. You’re visualizing fᵢ(xᵢ) for each feature.
'''
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Format function for x-axis
def currency_formatter(x, pos):
    if x >= 1_000_000:
        return f'${x / 1_000_000:.1f}M'
    elif x >= 1_000:
        return f'${x / 1_000:.1f}K'
    else:
        return f'${x:.0f}'

fig, axs = plt.subplots(2, 3, figsize=(20, 13))
for i, ax in enumerate(axs.flatten()):
    XX = gam.generate_X_grid(term=i)
    ax.plot(XX[:, i], gam.partial_dependence(term=i, X=XX))

    # Set custom x-axis formatter
    ax.xaxis.set_major_formatter(FuncFormatter(currency_formatter))

    # Set axis labels
    ax.set_xlabel("Spending")
    ax.set_ylabel("GA's")

    # Set subplot title to variable name
    ax.set_title(df.columns[i + df.columns.get_loc('paid_social')])

plt.tight_layout()
plt.show()

In [ ]:
# --- GPR

X = df[['days_since', 'paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']].values
y = df['ga'].values


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

'''
 Specifying constant kernel 1 means the model learns output variance from overall function
 Length scale specification uses one length per input variable, allowing for flexible smoothness between features
 Length scale bounds between -10 and 10 prevent model swinging highly to force fit
 Small alpha adds random noise to matrix diagonal. Helps to avoid overfitting without tainting data
'''
kernel = C(1.0) * RBF(length_scale=[1.0]*X.shape[1], length_scale_bounds=(0.01, 5e2))
gpr = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True, n_restarts_optimizer=10)
gpr.fit(X_scaled, y)

print(f"Kernel {gpr.kernel_}")
print(f"Log Marginal Likelihood {gpr.log_marginal_likelihood_value_}")

# Printing more comprehensive model output

constant_value = gpr.kernel_.k1.constant_value
print(f"  - Constant Value (Signal Variance): {constant_value}")


In [ ]:
# --- Fitted vs. Actual GPR
y_pred, y_std = gpr.predict(X_scaled, return_std=True)

plt.figure(figsize=(8,8))
plt.scatter(y, y_pred, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2, color = 'orange')
plt.xlabel('True Sales')
plt.ylabel('Predicted Sales (GPR Mean)')
plt.title('True vs Predicted')
plt.grid(True)
plt.show()

In [ ]:
# --- ICE + PDP - GPR
feature_names = ['Days_since', 'Paid Social', 'Paid Search', 'Paid Shopping', 'Email', 'Local Marketing', 'Programmatic']

fig, ax = plt.subplots(figsize=(20, 12))

# Capture the result into disp
disp = PartialDependenceDisplay.from_estimator(  # <<< assign it to disp!
    gpr,
    X_scaled,
    features=[0, 1, 2, 3, 4, 5],
    kind="both",
    grid_resolution=2500,
    ax=ax
)

# Now disp exists, and you can loop over disp.axes_
for axi, title in zip(disp.axes_.ravel(), feature_names):
    axi.set_title(title, fontsize=14, fontweight='bold', pad=10)

# Red PDP line
for ax in disp.axes_.ravel():
    for line in ax.get_lines():
        if line.get_label() == 'average':
            line.set_color('orange')
            line.set_linewidth(1.5)

plt.tight_layout()
plt.show()


In [ ]:
# --- Random Forest

X = df[['days_since','paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']].values
y = df['ga'].values

rf = RandomForestRegressor(
    n_estimators=500,
    max_features='sqrt',
    oob_score=True,
    random_state=42
)

rf.fit(X, y)

print(f"Training R²: {rf.score(X, y):.3f}")
print(f"OOB Score: {rf.oob_score_:.3f}")
print("Feature Importances:", rf.feature_importances_)


In [ ]:
# --- RF Feature Importance
importances = rf.feature_importances_
feature_names = ['days_since', 'paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']

# Sort importances
indices = np.argsort(importances)

# Plot
fig, ax = plt.subplots(figsize=(8,6))
ax.barh(range(len(importances)), importances[indices], align='center')
ax.set_yticks(range(len(importances)))
ax.set_yticklabels(np.array(feature_names)[indices])
ax.set_xlabel('Relative Importance')
ax.set_title('Random Forest Feature Importances')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# --- PDP + ICE - RF
pdp_display = PartialDependenceDisplay.from_estimator(
    rf,
    X,  # raw X is fine
    features=[0, 1, 2, 3, 4, 5, 6],
    feature_names=feature_names,
    grid_resolution=50,
    kind="both"
)

# Adjust the figure
pdp_display.figure_.suptitle('Partial Dependence - Random Forest', fontsize=20)
pdp_display.figure_.set_size_inches(14, 10)
plt.tight_layout(pad=3.0)
plt.show()

In [ ]:
# --- GBM

X = df[['days_since','paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']].values
y = df['ga'].values


gbm = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42
)

gbm.fit(X, y)

y_pred = gbm.predict(X)

cv_scores = cross_val_score(
    gbm, 
    X, 
    y, 
    cv=5,           # 5-fold CV
    scoring='r2'    # Use R² as the scoring metric
)

print(f"Cross-validated R² scores: {cv_scores}")

print("Feature Importances:", gbm.feature_importances_)
print(f"Training R²: {gbm.score(X, y):.3f}")
print(f"Mean CV R²: {np.mean(cv_scores):.3f}")
print(f"Cross-validated R² scores: {cv_scores}")

In [ ]:
# --- GBM Feature Importance
importances = gbm.feature_importances_
feature_names = ['days_since', 'paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']

# Sort importances
indices = np.argsort(importances)

# Plot
fig, ax = plt.subplots(figsize=(8,6))
ax.barh(range(len(importances)), importances[indices], align='center')
ax.set_yticks(range(len(importances)))
ax.set_yticklabels(np.array(feature_names)[indices])
ax.set_xlabel('Relative Importance')
ax.set_title('Gradient Boosting Feature Importances')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# --- PDP + ICE - GBM
pdp_display = PartialDependenceDisplay.from_estimator(
    gbm,
    X,
    features=[0, 1, 2, 3, 4, 5, 6],
    feature_names=feature_names,
    kind="both",  # both ICE curves and PDP average
    grid_resolution=1000  # high resolution for smoother look
)

# Adjust figure
pdp_display.figure_.suptitle('GMB - Ice / PDP', fontsize=20)
pdp_display.figure_.set_size_inches(14, 10)
plt.tight_layout(pad=3.0)
plt.show()


In [ ]:
# --- MLP - Shallow Neural Networks
import optuna
from skopt.space import Real, Categorical, Integer
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import clone
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor



X = df[['days_since', 'paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']].values
y = df['ga'].values

# Train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# SKL pipeline w/ 10k max iterations & Standard Scaler
pipe = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        max_iter=10_000,
        early_stopping=True,
        random_state=42,
        verbose=False
    )
)


def objective(trial):
    # Suggest hyperparameters
    hidden = trial.suggest_categorical('mlpregressor__hidden_layer_sizes', [(32,), (64,)])
    alpha = trial.suggest_loguniform('mlpregressor__alpha', 1e-5, 1e-2)
    lr    = trial.suggest_loguniform('mlpregressor__learning_rate_init', 1e-4, 1e-2)

    model = clone(pipe)
    model.set_params(mlpregressor__hidden_layer_sizes=hidden,
                     mlpregressor__alpha=alpha,
                     mlpregressor__learning_rate_init=lr)
    # 5-fold CV on the training pool
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
    return scores.mean()


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("Best params:", study.best_params)

best_model = clone(pipe)
best_model.set_params(**study.best_params)
best_model.fit(X_train, y_train)

print("Test R²:", best_model.score(X_test, y_test))

In [ ]:
# --- Loss plot for MLP
mlp = best_model.named_steps['mlpregressor']
plt.figure(figsize=(6,4))
plt.plot(mlp.loss_curve_)
plt.xlabel("Iteration")
plt.ylabel("Training Loss")
plt.title("MLP Training Loss Curve")
plt.grid(True)
plt.show()

In [ ]:
# --- ICE + PDP - Shallow MLP

features = ['Days Since','Paid Social', 'Paid Search', 'Paid Shopping', 'Email', 'Local Marketing', 'Programmatic']

mlppdp = PartialDependenceDisplay.from_estimator(
    best_model,
    X_train,
    features=[0, 1, 2, 3, 4, 5, 6],
    feature_names= features,
    kind='both',       # 'both' for ICE + PDP
    subsample=200,     # to speed up ICE plotting
    n_jobs=-1
)

for ax, name in zip(disp.axes_.ravel(), features):
    ax.set_title(f"{name} PDP", fontsize=12)

mlppdp.figure_.suptitle('Shallow MLP - Ice / PDP', fontsize=20)
mlppdp.figure_.set_size_inches(14, 10)

plt.tight_layout(pad=3.0)
plt.show()

In [ ]:
# --- Deep Neural Network 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import optuna

X = df[['days_since', 'paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']].values
y = df['ga'].values

# Train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ============================================
# 1. Deep Neural Net Implementation (PyTorch)
# ============================================

class MLPNet(nn.Module):
    """
    Flexible feedforward network.
    hidden_dims: list of int, sizes of successive hidden layers.
    dropout: float, dropout probability between layers.
    """
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout=0.1):
        super().__init__()
        layers = []
        prev_dim = input_dim
        # build each hidden layer
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h
        # final output layer
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_pytorch_mlp(
    X_train, y_train,
    X_val,   y_val,
    hidden_dims=[128,64,32],
    lr=1e-3,
    batch_size=32,
    epochs=1000,
    patience=20,
    threshold=1e-4,
    dropout=0.1,
    weight_decay=0.0,
    device=None,
    seed=42
):
# Set reproduceability seed, offer optional cuda, early stop on CV validation hold-out
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

    # build datasets/loaders
    X_tr = torch.tensor(X_train, dtype=torch.float32)
    y_tr = torch.tensor(y_train, dtype=torch.float32)
    X_v  = torch.tensor(X_val,   dtype=torch.float32)
    y_v  = torch.tensor(y_val,   dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(X_tr, y_tr),
        batch_size=batch_size, shuffle=True, pin_memory=(device!='cpu')
    )
    val_loader = DataLoader(
        TensorDataset(X_v, y_v),
        batch_size=batch_size, shuffle=False, pin_memory=(device!='cpu')
    )

    # model / opt / loss
    model = MLPNet(input_dim=X_train.shape[1], hidden_dims=hidden_dims, dropout=dropout)
    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_state    = None
    wait = 0


    # Training loop
    for epoch in range(1, epochs+1):
        # --- training pass ---
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        # --- validation pass ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_loss += criterion(model(xb), yb).item() * xb.size(0)
        val_loss /= len(val_loader.dataset)

        # early stopping check
        if best_val_loss - val_loss > threshold:
            best_val_loss = val_loss
            best_state   = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"⏹️ Early stop at epoch {epoch} (val_loss plateaued at {best_val_loss:.4f})")
                break

        if epoch % 10 == 0:
            print(f"Epoch {epoch:04d} | val_loss: {val_loss:.4f}")

    # restore best weights if we stopped early
    if best_state is not None:
        model.load_state_dict(best_state)

    return model


def cv_score_pytorch(
    X, y,
    cv=5,
    train_fn=train_pytorch_mlp,
    model_kwargs=None,
    device=None,
    trial=None,
):
    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = []
    model_kwargs = model_kwargs or {}

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X), 1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        # scale features **once** per fold
        scaler = StandardScaler().fit(X_tr)
        X_tr, X_va = scaler.transform(X_tr), scaler.transform(X_va)
        # train with validation set for early stopping
        model = train_fn(
            X_tr, y_tr,
            X_va, y_va,
            device=device,
            **model_kwargs
        )

        # final R² on fold’s hold-out
        model.eval()
        with torch.no_grad():
            preds = model(torch.tensor(X_va, dtype=torch.float32).to(device))
            y_pred = preds.cpu().numpy()
        fold_score = r2_score(y_va, y_pred)
        scores.append(fold_score)

        # Optuna reporting & pruning
        if trial is not None:
            trial.report(fold_score, step=fold)
            if trial.should_prune():
                raise optuna.TrialPruned()

    return float(np.mean(scores))


cfg = {
    'hidden_dims': [128, 64, 32],
    'lr'         : 1e-3,
    'batch_size': 32,
    'epochs'    : 1000,
    'dropout'   : 0.1,
}

scores = cv_score_pytorch(
    X_train, y_train,
    cv=5,
    train_fn=train_pytorch_mlp,
    model_kwargs=cfg,
    device=None
)
print("Mean CV R²:", scores)

study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)

def objective(trial):
    # 1) dynamically build hidden_dims
    n_layers = trial.suggest_int("n_layers", 1, 4)
    hidden_dims = [
        trial.suggest_int(f"n_units_l{i}", 32, 256, step=32)
        for i in range(n_layers)
    ]

    cfg = {
        "hidden_dims"   : hidden_dims,
        "lr"            : trial.suggest_loguniform("lr", 1e-4, 1e-2),
        "batch_size"    : trial.suggest_categorical("batch_size", [16,32,64,128]),
        "dropout"       : trial.suggest_float("dropout", 0.0, 0.5),
        "weight_decay"  : trial.suggest_loguniform("weight_decay", 1e-6, 1e-2),
    }
    return cv_score_pytorch(X_train, y_train, model_kwargs=cfg, trial=trial)

study.optimize(objective, n_trials=30, timeout=3600)

# ── FINAL HOLD-OUT EVAL ────────────────────────────────────────────

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import torch


In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin

class TorchRegressor(BaseEstimator, RegressorMixin):
    _estimator_type = "regressor"    # ← this tells is_regressor to pass

    def __init__(self, torch_model, device="cpu"):
        self.torch_model = torch_model
        self.device = device

    def fit(self, X, y=None):
        X = np.asarray(X)
        self.n_features_in_ = X.shape[1]
        self.is_fitted_ = True        # mark as fitted
        return self

    def predict(self, X):
        import torch
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        self.torch_model.eval()
        with torch.no_grad():
            return self.torch_model(X_t).cpu().numpy()

# 1) Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# 2) Scale full train/test splits
scaler   = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

# 3) Train on all training data with best hyperparams
best = study.best_params
n_layers = best['n_layers']
hidden_dims = [best[f'n_units_l{i}'] for i in range(n_layers)]
final_model = train_pytorch_mlp(
    X_train_s, y_train,
    X_test_s,  y_test,
    hidden_dims  = hidden_dims,
    lr           = best['lr'],
    batch_size   = best['batch_size'],
    dropout      = best['dropout'],
    weight_decay = best['weight_decay'],
    device       = DEVICE
)

# 4) Evaluate on the hold-out test set
final_model.eval()
with torch.no_grad():
    preds_test = final_model(
        torch.tensor(X_test_s, dtype=torch.float32).to(DEVICE)
    ).cpu().numpy()

print("Final Test R²:", r2_score(y_test, preds_test))

sk_model = TorchRegressor(final_model)
sk_model.fit(X_train_s, y_train)

from sklearn.inspection import PartialDependenceDisplay

# feature indices and names
features     = [1,2,3,4,5,6]
feature_names = ['Paid Social','Paid Search','Paid Shopping','Email Marketing','Local Marketing','Display, Audio, and Programmatic']

# decide grid resolution and percentile bounds
grid_resolution = 50
low, high = 5, 95

fig, axes = plt.subplots(2, 3, figsize=(12,6))
axes = axes.flatten()

for ax, feat_idx, feat_name in zip(axes, features, feature_names):
    # 1) build the grid for this feature
    col = X_train_s[:, feat_idx]
    grid = np.linspace(np.percentile(col, low),
                       np.percentile(col, high),
                       grid_resolution)
    
    # 2) for each grid value, swap it into a copy of X and predict
    pdp = []
    X_temp = X_train_s.copy()
    for val in grid:
        X_temp[:, feat_idx] = val
        preds = sk_model.predict(X_temp)
        pdp.append(preds.mean())
    
    # 3) plot
    ax.plot(grid, pdp, lw=2)
    ax.set_title(feat_name)
    ax.set_xlabel(feat_name)
    ax.set_ylabel("Predicted ga")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# 2. Symbolic Regression (PySR)
# ============================================
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd

X = df[['days_since', 'paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']]
y = df['ga']

# Train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

feature_names = ['days_since', 'paid_social', 'paid_search', 'paid_shopping', 'email', 'local', 'program']

model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*", "-", "/"],
    unary_operators=["exp", "log", "sqrt", "sin", "cos"],
    model_selection="best",  # or "accuracy", "complexity"
    maxsize=20,
    loss="loss(x, y) = (x - y)^2",
    population_size=1000,
    warm_start=False,
    progress=True,
    verbosity=1,
    #feature_names=feature_names,
    random_state=42,
    deterministic=True, 
    parallelism='serial'
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Test R²: {r2:.4f}")
print(f"Test RMSE: {rmse:.2f}")

In [ ]:
print(dir(model))
frontier = model.equations_
print(frontier)

frontier = pd.DataFrame(frontier)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(frontier["complexity"], frontier["loss"], c="tab:blue")
plt.xlabel("Complexity")
plt.ylabel("Loss")
plt.title("Pareto Frontier: Complexity vs. Loss")
plt.show()

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def plot_frontier_pdps(frontier_df, X, feature_names=None, npts=200):
    """
    frontier_df : pd.DataFrame 
        Must have columns 'sympy_format', 'loss', 'score'.
    X : pandas.DataFrame or ndarray
        Your training features.  If ndarray, you MUST supply feature_names.
    feature_names : list[str], optional
        Names of the columns in X, in the same order the model saw them.
    npts : int
        Resolution of each 1D slice.
    """
    # 1) Coerce X into a DataFrame if it isn’t one already
    if not isinstance(X, pd.DataFrame):
        if feature_names is None:
            raise ValueError("When passing an ndarray for X, you must also pass feature_names")
        X = pd.DataFrame(X, columns=feature_names)
    
    medians = X.median()
    mins     = X.min()
    maxs     = X.max()

    for idx, row in frontier_df.iterrows():
        expr = sp.sympify(row["sympy_format"])
        syms = sorted(expr.free_symbols, key=lambda s: str(s))
        
        # 2) Skip constant formulas
        if len(syms) == 0:
            print(f"Skipping constant equation: {row['sympy_format']}")
            continue
        
        # 3) If your sympy symbols are named x0, x1, …, map them to your real columns
        #    (only needed if you didn’t pass feature_names into PySRRegressor)
        if feature_names is not None and all(str(s).startswith("x") for s in syms):
            mapping = { s: sp.symbols(feature_names[int(str(s)[1:])]) for s in syms }
            expr = expr.xreplace(mapping)
            syms = sorted(expr.free_symbols, key=lambda s: str(s))
            fn = sp.lambdify(syms, expr, "numpy")
        else:
            fn = sp.lambdify(syms, expr, "numpy")

        # 4) Build one subplot per symbol
        n_vars = len(syms)
        fig, axes = plt.subplots(1, n_vars, figsize=(4*n_vars, 3), squeeze=False)
        fig.suptitle(
            f"{expr}\n"
            f"loss={row['loss']:.2e}, score={row['score']:.3f}",
            fontsize=12,
        )

        for j, sym in enumerate(syms):
            name = str(sym)
            lo, hi = mins[name], maxs[name]
            x_vals = np.linspace(lo, hi, npts)

            # assemble full input matrix, holding others at median
            inputs = np.vstack([
                x_vals if s == sym else np.full(npts, medians[str(s)])
                for s in syms
            ]).T

            y_vals = fn(*[inputs[:, k] for k in range(inputs.shape[1])])

            ax = axes[0, j]
            ax.plot(x_vals, y_vals, lw=2, alpha=0.8)
            ax.set_xlabel(name)
            ax.set_ylabel("f")
            ax.set_title(name)

        plt.tight_layout(rect=[0, 0.03, 1, 0.90])
        plt.show()


In [ ]:
features = ['days_since', 'paid_social','paid_search','paid_shopping',
            'email','local','program']

# make X a DataFrame so .min()/.median() work
X = df[features]
frontier = pd.DataFrame(model.equations_)

plot_frontier_pdps(frontier, X, feature_names=features)


In [ ]:
df.reset_index().to_feather("df_export.feather")  # if you want to keep the index